# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de Gemini

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [9]:
import google.generativeai as genai

In [ ]:
GEMINI_API_KEY = ""

In [11]:
genai.configure(api_key=GEMINI_API_KEY)

model_gemini = genai.GenerativeModel('gemini-3.5-flash')

response = model_gemini.generate_content("Quiero que me des un resumen de la novela 'Cien años de soledad' de Gabriel García Márquez")
print(response.text)

**"Cien años de soledad"** (1967), escrita por el colombiano Gabriel García Márquez (Premio Nobel de Literatura), es una de las obras cumbres de la literatura hispanoamericana y mundial, y la máxima representante del **realismo mágico**. 

Aquí tienes un resumen estructurado de la novela, que narra la historia de siete generaciones de la familia Buendía y del pueblo de Macondo.

---

### 1. La Fundación de Macondo (El Origen)
La historia comienza con **José Arcadio Buendía** y **Úrsula Iguarán**, una pareja de primos que se casan en su pueblo natal. Debido al incesto, temen tener un hijo con cola de cerdo. Tras un duelo de honor en el que José Arcadio mata a un hombre (Prudencio Aguilar), el fantasma de este lo atormenta, obligando a la pareja y a varios amigos a cruzar la sierra en busca de un nuevo hogar. Así fundan **Macondo**, un pueblo utópico, aislado del mundo y rodeado de agua.

En sus inicios, Macondo es un lugar casi mágico donde las cosas no tienen nombre. El pueblo recibe v

## 2. Retrieval
### 2.1 Cargo el corpus de 20 News Groups

In [13]:
from sklearn.datasets import fetch_20newsgroups

print("Descargando...")
newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
docs = newsgroups.data
print(f"Listo! Documentos cargados: {len(docs)}")

Descargando...
Listo! Documentos cargados: 18846


### 2.2 Transformo a embeddings

In [14]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

docs_subset = docs[:1000]

print(f"Generando embeddings para {len(docs_subset)} documentos...")
embeddings = model.encode(docs_subset, show_progress_bar=True)
print(f"Listo! Shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generando embeddings para 1000 documentos...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Listo! Shape: (1000, 384)


### 2.3 Creo una query y hago la búsqueda

In [25]:
from sklearn.metrics.pairwise import cosine_similarity

query = "What are commercial activities in space?"
query_emb = model.encode(query)

similarities = cosine_similarity(query_emb.reshape(1, -1), embeddings)

In [26]:
top_k = 5
top_indices = similarities[0].argsort()[-top_k:][::-1]

for i, idx in enumerate(top_indices):
    print(f"\n--- Documento {idx} (similitud: {similarities[0][idx]:.4f}) ---")
    print(docs_subset[idx][:300])


--- Documento 847 (similitud: 0.5594) ---
Not to mention how those those liberal presidents, Nixon, Ford,
Reagan, Bush.   did nothing to support  true commercial space
activities.

--- Documento 337 (similitud: 0.4723) ---
Original to: szabo@techbook.com
G'day szabo@techbook.com

29 Mar 93 07:28, szabo@techbook.com wrote to All:

 sc> szabo@techbook.com (Nick Szabo), via Kralizec 3:713/602

 sc> Here are some longer-term markets to consider:

Here are some more:

* Terrestrial illumination from orbiting mirrors.

* Wo

--- Documento 784 (similitud: 0.4601) ---

Whatabout, Schools, Universities, Rich Individuals (around 250 people 
in the UK have more than 10 million dollars each). I reecieved mail
from people who claimed they might get a person into space for $500
per pound. Send a skinny person into space and split the rest of the money
among the ground 

--- Documento 390 (similitud: 0.4504) ---
As for SF and advertising in space. There is a romantic episode
in Mead's "The Big Ball 

In [ ]:
context = "\n\n".join([docs_subset[idx][:500] for idx in top_indices])

prompt = f"""Eres una aplicación de Retrieval Augmented Generation que siempre responde en español. Usa el siguiente contexto para responder la pregunta. 
Si la respuesta no está en el contexto, di que no sabes.

Contexto:
{context}

Pregunta:
El usuario está preguntando sobre: {query}
"""

response = model_gemini.generate_content(prompt, temperature=0.3)
print(response.text)

Basándome en el contexto proporcionado, las actividades comerciales (o mercados a considerar) en el espacio incluyen las siguientes:

* **Iluminación terrestre** a partir de espejos en órbita.
* **Sistema de monitoreo de desastres y del medio ambiente mundial** (como el plan WEDOS desarrollado por los japoneses, aunque se menciona que podría ser más un "bien público").
* **Turismo espacial**.
* **Satélites de retransmisión de energía**.
* **Envío de personas al espacio** financiado por escuelas, universidades o personas adineradas (por un costo estimado de $500 por libra).
* **Publicidad en el espacio** (como la idea de poner en órbita una formación de espejos espaciales).


### 2.4 Comparación: LLM directo vs RAG

La ventaja del RAG es que la respuesta se fundamenta en documentos reales del corpus, 
mientras que el LLM solo puede responder con lo que aprendió en entrenamiento. 
Esto reduce alucinaciones y permite trabajar con información privada o especializada.

In [ ]:
# sin contexto, solo el LLM
response_directo = model_gemini.generate_content(query)
print("=== Respuesta SIN contexto (LLM directo) ===")
print(response_directo.text)

print("\n=== Respuesta CON contexto (RAG) ===")
response_rag = model_gemini.generate_content(prompt, generation_config={"temperature": 0.3})
print(response_rag.text)

=== Respuesta SIN contexto (LLM directo) ===
Commercial activities in space—often referred to as the **"NewSpace" economy**—encompass any business or industrial activity that takes place in outer space, or directly supports space operations from Earth. 

Historically, space was the exclusive domain of national governments (like NASA or the Soviet space program). Today, private companies drive a rapidly growing global space economy estimated to reach **$1 trillion by 2040**.

Here is a comprehensive breakdown of the key commercial activities in space, categorized from current mature markets to emerging and future frontiers.

---

### 1. Satellite Communications (The Largest Sector)
This is currently the most mature and lucrative sector of the commercial space economy. 
*   **Satellite Internet:** Companies deploy constellations of thousands of small satellites in Low Earth Orbit (LEO) to provide high-speed internet to remote or underserved areas. *(Examples: SpaceX’s Starlink, Eutelsat 